In [ ]:
from math import sqrt

import numpy as np
import pandas as pd
import seaborn as sb
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
from scipy.stats import chi2
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif

G = np.random.default_rng()

In [ ]:
data = pd.read_excel("../data1/cps12.xlsx")
datam = pd.read_excel("../data1/CollegeDistance.xls")

In [ ]:
data.head()

In [ ]:
datam.head()

# Гетероскедастичность

In [ ]:
model = sm.OLS.from_formula("ahe ~ age", data=data)
model_est = model.fit()
print(model_est.summary())

In [ ]:
model.exog

In [ ]:
resid_se = sqrt(model_est.ssr / model_est.df_resid)

## Тестирование

### Метод вдумчивого взгляда

Графики стандартизованных остатков от:

* номера наблюдения,
* `age`,
* `ahe`.

In [ ]:
sb.scatterplot(x=range(len(model_est.resid)), y=(model_est.resid / resid_se))

In [ ]:
sb.scatterplot(data, y=(model_est.resid / resid_se), x="age")

In [ ]:
sb.scatterplot(data, y=(model_est.resid / resid_se), x="ahe")

### Тест Голдфельдта-Квандта

Принимает на вход:

* `y` --- матрицу значений **объясняемой** переменной;
* `x` --- матрицу значений **объясняющих** переменных;
* `idx` --- номер столца интересующей нас переменной из `x`;
* `drop` --- число (или доля) отбрасываемых наблюдений;
* `split` --- доля наблюдений, идущих в **первую** подвыборку.

In [ ]:
sms.het_goldfeldquandt(
    y=model.endog,
    x=model.exog,
    idx=1,
    drop=0.3,
    split=0.35,
    result_object=True,
)

### Тест Бройша-Пагана

$$\begin{aligned}
y & = X \beta + u \\
\hat{u}^2 & = \gamma_0 + Z \gamma + \nu \\
nR^2 & \sim \chi^2(p) \\
\quad F & \sim F(n - 1, n - k - 1)
\end{aligned}$$

На вход принимает остатки и матрицу объясняющих переменных вспомогательной регрессии.

In [ ]:
sms.het_breuschpagan(
    resid=model_est.resid,
    exog_het=model.exog,
    robust=True,
)

### Тест Уайта

$$\begin{aligned}
y & = X \beta + u \\
\hat{u}^2 & = \gamma_0 + \sum_i \gamma_i z_i + \sum_i \sum_{j \geq i} \gamma_{ij} z_i z_j + \nu \\
nR^2 & \sim \chi^2(p) \\
\quad F & \sim F(n - 1, n - k - 1)
\end{aligned}$$

На вход принимает остатки и матрицу объясняющих переменных вспомогательной регрессии.

In [ ]:
sms.het_white(
    resid=model_est.resid,
    exog=model.exog,
    interaction_terms=True,
)

### Тест Глейзера

С ним есть определенные сложности в том плане, что его реализации несколько отличаются от классического описания из учебников. После оценивания модели и получения модуля остатков $|\hat{u}|$ оценивается вспомогательная модель
$$|\hat{u}| = \beta_0 + Z'\beta + \nu$$
где $Z$ --- объясняющие переменные вспомогательного уравнения. Часто в качестве $Z$ берут переменные исходной модели. Тест имеет две статистики:
$$\begin{aligned}nR^2 & \sim \chi^2(k) \\
\frac{ESS_{aux}}{\left(1 - \frac{2}{\pi}\right)\hat\sigma^2} & \sim \chi^2(k)
\end{aligned}$$
где $ESS_{aux}$ --- объясненная сумма квадратов вспомогательной модели, $\hat\sigma^2$ --- выборочная дисперсия остатков исходной или вспомогательной модели.

In [ ]:
model_aux = smf.ols("abs(model_est.resid) ~ age", data=data)
model_aux_est = model_aux.fit()
stat_nR2 = model_aux_est.nobs * model_aux_est.rsquared
stat_aux = model_aux_est.ess / ((1 - 2 / np.pi) * np.var(model_est.resid))
print(
    f"Stat: {stat_nR2:5.4f}, Critical value: {chi2.ppf(0.95, df=model_aux.df_model):5.4f}, p-value: {1 - chi2.cdf(stat_nR2, df=model_aux.df_model):5.4f}"
)
print(
    f"Stat: {stat_aux:5.4f}, Critical value: {chi2.ppf(0.95, df=model_aux.df_model):5.4f}, p-value: {1 - chi2.cdf(stat_aux, df=model_aux.df_model):5.4f}"
)

## Моделирование

ВМНК своими руками:

In [ ]:
print(
    sm.OLS.from_formula(
        "I(ahe / np.sqrt(age)) ~ -1 + I(1/np.sqrt(age)) + np.sqrt(age)", data=data
    )
    .fit()
    .summary()
)

ВМНК без велосипедостроения:

In [ ]:
print(sm.WLS.from_formula("ahe~age", data=data, weights=(1 / data.age)).fit().summary())

In [ ]:
model_aux = sm.OLS(model_est.resid**2, model.exog).fit()
fitted_u = model_aux.fittedvalues

In [ ]:
print(sm.WLS.from_formula("ahe~age", data=data, weights=(1 / fitted_u)).fit().summary())

### Робастные стандартные ошибки

In [ ]:
model_est.HC1_se

In [ ]:
print(model_est.get_robustcov_results(cov_type="HC0").summary())

In [ ]:
model = sm.OLS.from_formula("ahe ~ age", data=data)
model_fitted = model.fit(cov_type="HC0")
print(model_fitted.summary())

# Автокорреляция

In [ ]:
data_ac = pd.DataFrame({"x": G.normal(size=150)})

rho = 0.5
nu = G.normal(scale=1, size=150)

u = np.zeros(151)
u[0] = G.normal()
for t in range(150):
    u[t + 1] = rho * u[t] + nu[t]
u = u[1:]

data_ac["y"] = 1 + 1 * data_ac.x + u

In [ ]:
sb.lineplot(x="x", y="y", data=data_ac)

In [ ]:
model = sm.OLS.from_formula("y~x", data=data_ac)
model_est = model.fit()
print(model_est.summary())

In [ ]:
sms.durbin_watson(model_est.resid)

In [ ]:
sms.acorr_breusch_godfrey(
    model_est,
    result_object=True,
)

In [ ]:
sms.acorr_ljungbox(model_est.resid, model_df=model_est.df_model)

### Процедура Кохрейна-Оркатта

In [ ]:
uhat = np.array(model_est.resid)[:, np.newaxis]
model_aux = sm.OLS(
    uhat[1:, :],
    uhat[:-1, :],
).fit()
rho_hat = model_aux.params.item()

In [ ]:
rho_hat

In [ ]:
yy = data_ac.y - rho_hat * data_ac.y.shift(1)
yy[0] = sqrt(1 - rho_hat**2) * data_ac.y[0]

xx = data_ac.x - rho_hat * data_ac.x.shift(1)
xx[0] = sqrt(1 - rho_hat**2) * data_ac.x[0]

cc = np.full_like(yy, fill_value=1 - rho_hat)
cc[0] = sqrt(1 - rho_hat**2)

In [ ]:
data_ac["yy"] = yy
data_ac["xx"] = xx
data_ac["cc"] = cc

In [ ]:
model_co = sm.OLS.from_formula("yy~cc+xx-1", data=data_ac)
model_co_est = model_co.fit()
print(model_co_est.summary())

In [ ]:
sms.durbin_watson(model_co_est.resid)

In [ ]:
sms.acorr_breusch_godfrey(
    model_co_est,
    result_object=True,
)

Разумеется, что того же эффекта можно добиться просто попросив HAC-оценку ковариационной матрицы.
При этом нам потребуется указать ширину окна (`maxlags`) для оценки.

In [ ]:
print(model_est.get_robustcov_results(cov_type="HAC", maxlags=5).summary())

# Мультиколлинеарность

In [ ]:
model = smf.ols(
    "ed ~ dist + bytest + female + black + hispanic + incomehi + ownhome + dadcoll + cue80 + stwmfg80",
    data=datam,
)
model_est = model.fit()
print(model_est.summary())

In [ ]:
for i in range(1, model.exog.shape[1]):
    vif_est = vif(model.exog, i)
    print(f"{model_est.params.index[i]:8}: {vif_est:5.4f}")

In [ ]:
datam["bytest_"] = datam.bytest + G.normal(size=3796, scale=2)

In [ ]:
datam[["bytest", "bytest_"]].corr()

In [ ]:
model3 = smf.ols(
    "ed ~ dist + bytest + bytest_ + female + black + hispanic + incomehi + ownhome + dadcoll + cue80 + stwmfg80",
    data=datam,
)
model3_est = model3.fit()
print(model3_est.summary())

In [ ]:
model3_est.f_test("bytest = 0, bytest_ = 0")

In [ ]:
for i in range(1, model3.exog.shape[1]):
    vif_est = vif(model3.exog, i)
    print(f"{model3_est.params.index[i]:8}: {vif_est:5.4f}")

In [ ]:
datam["male"] = 1 - datam.female

In [ ]:
model4 = smf.ols(
    "ed ~ dist + bytest + female + male + black + hispanic + incomehi + ownhome + dadcoll + cue80 + stwmfg80",
    data=datam,
)
model4_est = model4.fit()
print(model4_est.summary())

In [ ]:
for i in range(1, model4.exog.shape[1]):
    vif_est = vif(model4.exog, i)
    print(f"{model4_est.params.index[i]:8}: {vif_est:5.4f}")

In [ ]:
datam["bytest__"] = 1 + 30 * datam.bytest

In [ ]:
datam[["bytest", "bytest__"]].corr()

In [ ]:
model5 = smf.ols(
    "ed ~ dist + bytest + bytest__ + female + black + hispanic + incomehi + ownhome + dadcoll + cue80 + stwmfg80",
    data=datam,
)
model5_est = model5.fit()
print(model5_est.summary())

In [ ]:
for i in range(1, model5.exog.shape[1]):
    vif_est = vif(model5.exog, i)
    print(f"{model5_est.params.index[i]:8}: {vif_est:5.4f}")